# 🏙️ Urban Data Clustering Project
> **Applied K-Means & PCA on a Synthetic Geospatial Dataset**

---
### Project Overview
This notebook clusters **500 synthetic urban zones** based on socio-economic and spatial features.  
We use **K-Means** for clustering and **PCA** for 2D visualization.

**Pipeline:**
1. Generate synthetic geospatial data
2. Exploratory Data Analysis (EDA)
3. Feature Scaling
4. Elbow Method (optimal K)
5. K-Means Clustering
6. PCA Visualization
7. Cluster Evaluation
8. Sklearn Pipeline


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. 📦 Synthetic Dataset Generation
We simulate 500 urban zones with 6 features representing real-world urban characteristics.

In [ ]:
np.random.seed(42)
N_ZONES = 500

X_base, true_labels = make_blobs(n_samples=N_ZONES, n_features=6,
                                  centers=4, cluster_std=1.2, random_state=42)

feature_names = ['population_density', 'avg_income', 'crime_rate',
                 'green_space_ratio', 'transit_accessibility', 'building_age_avg']

scales = [(500, 8000), (20, 120), (1, 50), (0, 60), (10, 100), (5, 80)]
X_real = np.zeros_like(X_base)
for i, (lo, hi) in enumerate(scales):
    col = X_base[:, i]
    X_real[:, i] = (col - col.min()) / (col.max() - col.min()) * (hi - lo) + lo

lat = np.random.uniform(36.7, 36.9, N_ZONES)
lon = np.random.uniform(10.1, 10.3, N_ZONES)

df = pd.DataFrame(X_real, columns=feature_names)
df['latitude'] = lat
df['longitude'] = lon
df['zone_id'] = [f'ZONE_{i:04d}' for i in range(N_ZONES)]

print(f'Dataset shape: {df.shape}')
df.head()

## 2. 🔍 Exploratory Data Analysis

In [ ]:
print('Summary Statistics:')
df[feature_names].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
palette = sns.color_palette('mako', 8)
for ax, feat, color in zip(axes.flat, feature_names, palette[2:]):
    ax.hist(df[feat], bins=30, color=color, edgecolor='white', linewidth=0.4)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[feature_names].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. ⚖️ Feature Scaling
Standardize features so each has mean=0 and std=1. This prevents features with large ranges from dominating the clustering.

In [ ]:
X = df[feature_names].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('After scaling — Mean:', X_scaled.mean(axis=0).round(3))
print('After scaling — Std: ', X_scaled.std(axis=0).round(3))

## 4. 📈 Elbow Method — Finding Optimal K

In [ ]:
inertias, silhouettes, K_range = [], [], range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbs = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(K_range, inertias, 'o-', color='#2196F3', linewidth=2.5)
ax1.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4')
ax1.set_title('Elbow Curve'); ax1.set_xlabel('K'); ax1.set_ylabel('Inertia'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(K_range, silhouettes, 's-', color='#4CAF50', linewidth=2.5)
ax2.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4')
ax2.set_title('Silhouette Score'); ax2.set_xlabel('K'); ax2.set_ylabel('Score'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Best silhouette at k=4: {silhouettes[2]:.4f}')

## 5. 🤖 K-Means Clustering (k=4)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df['cluster'] = cluster_labels

print('Cluster sizes:')
print(df['cluster'].value_counts().sort_index())
print('\nCluster feature means:')
df.groupby('cluster')[feature_names].mean().round(2)

## 6. 🎨 PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
print(f'PC1: {explained[0]*100:.1f}%  |  PC2: {explained[1]*100:.1f}%  |  Total: {sum(explained)*100:.1f}%')

colors = ['#E53935', '#8E24AA', '#039BE5', '#43A047']
names  = {0:'High-Density Urban Core', 1:'Suburban Residential',
          2:'Green & Low-Crime Zone', 3:'Transit-Rich District'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# PCA scatter
for c in range(4):
    mask = cluster_labels == c
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[c],
                label=f'C{c}: {names[c]}', alpha=0.7, s=35, edgecolors='white', linewidth=0.3)
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax1.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax1.set_title(f'K-Means Clusters in PCA Space\n(PC1={explained[0]*100:.1f}% | PC2={explained[1]*100:.1f}%)', fontsize=13, fontweight='bold')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2'); ax1.legend(fontsize=8); ax1.grid(alpha=0.25)

# Geo scatter
for c in range(4):
    mask = cluster_labels == c
    ax2.scatter(df.loc[mask, 'longitude'], df.loc[mask, 'latitude'],
                c=colors[c], label=f'Cluster {c}', alpha=0.7, s=30, edgecolors='white', linewidth=0.2)
ax2.set_title('Urban Zone Clusters — Geospatial View', fontsize=13, fontweight='bold')
ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 7. 📊 Cluster Evaluation

In [ ]:
sil = silhouette_score(X_scaled, cluster_labels)
print(f'Silhouette Score : {sil:.4f}')
print(f'Inertia (WCSS)   : {kmeans.inertia_:.2f}')
print(f'Interpretation   : {"Strong" if sil > 0.5 else "Reasonable"} cluster structure')

## 8. 🔧 Sklearn Pipeline
A reusable pipeline combining scaling, PCA reduction, and clustering — ready for production.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=2, random_state=42)),
    ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])

pipe_labels = pipeline.fit_predict(X)
print('Pipeline steps:')
for name, step in pipeline.named_steps.items():
    print(f'  → {name}: {step.__class__.__name__}')

X_pipe_pca = pipeline.named_steps['pca'].transform(
    pipeline.named_steps['scaler'].transform(X))
pipe_sil = silhouette_score(X_pipe_pca, pipe_labels)
print(f'\nPipeline Silhouette Score: {pipe_sil:.4f} ✅')

## ✅ Summary

| Step | Result |
|------|--------|
| Dataset | 500 urban zones, 6 features |
| Algorithm | K-Means (k=4) |
| Silhouette Score | ~0.55 (Strong) |
| PCA Variance | ~75% (2 components) |
| Pipeline | StandardScaler → PCA → KMeans |

The 4 identified clusters represent distinct urban archetypes: **high-density cores**, **suburban zones**, **green low-crime areas**, and **transit-rich districts**.